##**Cloud Risk Assessment Model**

## 0 - Install Dependencies & Imports + Risk Configuration

In [1]:
!pip install xgboost scikit-learn pandas numpy matplotlib pymysql sqlalchemy -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
import xgboost as xgb

print("All libraries imported successfully")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.2 MB/s eta 0:00:00
All libraries imported successfully


**Risk Weight Configuration**

In [2]:
# Run this cell to input risk weights interactively.
# All weights must sum to 100%.

print("-" * 60)
print("  RISK WEIGHT CONFIGURATION")
print("  Enter weights as percentages (e.g. 20 for 20%).")
print("  All weights must sum to 100.")
print("-" * 60)
print()

WEIGHT_PROMPTS = {
    "vulnerability_severity": "How severe are the vulnerabilities?     (mean_cvss, max_cvss, mean_epss)",
    "vulnerability_volume":   "How many vulnerabilities are there?     (open_vulns, total_vulns, density)",
    "active_exploitation":    "Are any actively exploited right now?   (KEV list, exploit ratio, EPSS risk)",
    "incident_history":       "What is the service's incident history? (incident count, severity, downtime)",
    "repository_health":      "How well is the codebase maintained?    (patch debt, security issues, age)",
    "cwe_attack_types":       "What attack types is it exposed to?     (CWE one-hot flags)",
}

RISK_WEIGHTS = {}
for key, description in WEIGHT_PROMPTS.items():
    while True:
        try:
            val = input(f"  {description}\n  → Weight for '{key}' (%): ").strip()
            val = float(val)
            if val < 0 or val > 100:
                print("  ⚠ Please enter a value between 0 and 100.\n")
                continue
            RISK_WEIGHTS[key] = val / 100
            print()
            break
        except ValueError:
            print("  ⚠ Please enter a number (e.g. 20).\n")

total = sum(RISK_WEIGHTS.values())
if abs(total - 1.0) > 0.001:
    print(f"⚠ WARNING: Weights sum to {total:.0%}, not 100%.")
    print(f"  Difference: {(total - 1.0) * 100:+.1f}%")
    print()
    auto_fix = input("  Auto-normalise to 100%? (y/n): ").strip().lower()
    if auto_fix == "y":
        RISK_WEIGHTS = {k: v / total for k, v in RISK_WEIGHTS.items()}
        print("  Weights normalised ✓")
    else:
        print("  ⚠ Proceeding with unbalanced weights — results may be unexpected.")

print()
print("-" * 60)
print("  THRESHOLD CONFIGURATION")
print("  Define what counts as 'high risk' for your organisation.")
print("-" * 60)
print()

THRESHOLD_PROMPTS = {
    "cvss_min":     ("Minimum CVSS score to flag as high risk",    7.0,  float),
    "kev_min":      ("Minimum CISA KEV vulnerabilities to flag",   1,    int),
    "epss_min":     ("Minimum average EPSS score to flag (0–1)",   0.1,  float),
    "incident_min": ("Minimum historical incidents to flag",        3,    int),
}

RISK_THRESHOLDS = {}
for key, (description, default, cast) in THRESHOLD_PROMPTS.items():
    while True:
        try:
            val = input(f"  {description}\n  → '{key}' (default: {default}): ").strip()
            if val == "":
                RISK_THRESHOLDS[key] = default
                print(f"  Using default: {default}\n")
            else:
                RISK_THRESHOLDS[key] = cast(val)
                print()
            break
        except ValueError:
            print(f"  ⚠ Please enter a valid number.\n")

print()
print("-" * 60)
print("  RISK BAND CONFIGURATION")
print("  Set the score boundaries for High / Medium / Low.")
print("-" * 60)
print()

BAND_PROMPTS = {
    "high_threshold":   ("Scores at or above this = HIGH risk",   0.70, float),
    "medium_threshold": ("Scores at or above this = MEDIUM risk", 0.40, float),
}

RISK_BANDS = {}
for key, (description, default, cast) in BAND_PROMPTS.items():
    while True:
        try:
            val = input(f"  {description}\n  → '{key}' (default: {default}): ").strip()
            if val == "":
                RISK_BANDS[key] = default
                print(f"  Using default: {default}\n")
            else:
                RISK_BANDS[key] = cast(val)
                print()
            break
        except ValueError:
            print(f"  ⚠ Please enter a value between 0 and 1 (e.g. 0.70).\n")

# Summary Report
print()
print("-" * 60)
print("  CONFIGURATION SUMMARY")
print("-" * 60)
print()
print("  CATEGORY WEIGHTS")
print("  " + "─" * 55)
for category, weight in RISK_WEIGHTS.items():
    bar = "█" * int(weight * 40)
    print(f"  {category:<25} {bar:<40} {weight:.0%}")
print(f"  {'─'*55}")
print(f"  {'TOTAL':<25} {sum(RISK_WEIGHTS.values()):.0%}")
print()
print("  RISK THRESHOLDS")
print("  " + "─" * 35)
for k, v in RISK_THRESHOLDS.items():
    print(f"  {k:<20} {v}")
print()
print("  RISK BANDS")
print("  " + "─" * 35)
print(f"  High   : score ≥ {RISK_BANDS['high_threshold']}")
print(f"  Medium : score ≥ {RISK_BANDS['medium_threshold']}")
print(f"  Low    : score <  {RISK_BANDS['medium_threshold']}")
print()
print("  Configuration complete.")


------------------------------------------------------------
  RISK WEIGHT CONFIGURATION
  Enter weights as percentages (e.g. 20 for 20%).
  All weights must sum to 100.
------------------------------------------------------------

  How severe are the vulnerabilities?     (mean_cvss, max_cvss, mean_epss)
  → Weight for 'vulnerability_severity' (%): 40

  How many vulnerabilities are there?     (open_vulns, total_vulns, density)
  → Weight for 'vulnerability_volume' (%): 25

  Are any actively exploited right now?   (KEV list, exploit ratio, EPSS risk)
  → Weight for 'active_exploitation' (%): 15

  What is the service's incident history? (incident count, severity, downtime)
  → Weight for 'incident_history' (%): 10

  How well is the codebase maintained?    (patch debt, security issues, age)
  → Weight for 'repository_health' (%): 5

  What attack types is it exposed to?     (CWE one-hot flags)
  → Weight for 'cwe_attack_types' (%): 5


------------------------------------------------

## 1 - Data Loading

In [3]:
import sqlalchemy as sa

engine = sa.create_engine(
    "mysql+pymysql://cloudriskdev:thisisyoursql1234!@"
    "mysql-cloudrisk-dev-west.mysql.database.azure.com:3306/cloudrisk",
    connect_args={"ssl": {"ssl_disabled": False}}
)

repo      = pd.read_sql("SELECT * FROM repo_health", engine)
incidents = pd.read_sql("SELECT * FROM service_incidents", engine)
services  = pd.read_sql("SELECT * FROM services", engine)
vulns_raw = pd.read_sql("SELECT * FROM vulnerabilities", engine)

print("Connected to Azure MySQL")

Connected to Azure MySQL


## 2 - Feature Engineering

 **2a. Rename columns**

In [4]:
vulns_raw.columns = [c if c != "id" else "service_id" for c in vulns_raw.columns]
repo.columns      = [c if c != "id" else "service_id" for c in repo.columns]
incidents.columns = [c if c != "id" else "service_id" for c in incidents.columns]

print("Column rename complete.")
print(f"  vulns_raw: {vulns_raw.columns.tolist()}")
print(f"  repo     : {repo.columns.tolist()}")
print(f"  incidents: {incidents.columns.tolist()}")


Column rename complete.
  vulns_raw: ['service_id', 'source', 'cve_id', 'advisory_id', 'service_name', 'ecosystem', 'severity', 'cvss_v3_score', 'cvss_v3_vector', 'epss_score', 'is_kev', 'cwe_id', 'published_at', 'modified_at', 'raw_json', 'ingested_at']
  repo     : ['service_id', 'service_name', 'repo_full_name', 'commit_count_90d', 'contributor_count_90d', 'open_issues', 'security_issues_open', 'mean_days_close_security', 'days_since_last_release', 'has_security_policy', 'raw_json', 'collected_at']
  incidents: ['service_id', 'service_name', 'incident_id', 'severity', 'status', 'started_at', 'resolved_at', 'duration_minutes', 'affected_components', 'raw_json', 'collected_at']


**2b.Aggregations**

In [5]:
services = (
    vulns_raw[["service_name", "ecosystem"]]
    .drop_duplicates(subset="service_name")
    .reset_index(drop=True)
)
services["vendor"]  = services["service_name"]
services["is_oss"]  = 0
services["is_saas"] = 0

print(f"Unique services: {len(services)}")

# Aggregating Incidents
sev_map = {"critical": 4, "high": 3, "medium": 2, "low": 1}
incidents["sev_score"] = incidents["severity"].map(sev_map).fillna(0)
incidents["resolved"]  = (incidents["status"] == "resolved").astype(int)

incidents_agg = incidents.groupby("service_name").agg(
    incident_count  = ("incident_id",      "count"),
    avg_severity    = ("sev_score",        "mean"),
    recurrence_rate = ("resolved",         lambda x: (1 - x).mean()),
    total_downtime  = ("duration_minutes", "sum"),
).reset_index()

# Aggregating Vulnerabilities
vuln_agg = vulns_raw.groupby("service_name").agg(
    total_vulns   = ("cve_id",        "count"),
    open_vulns    = ("severity",      lambda x: x.isin(["critical", "high"]).sum()),
    mean_cvss     = ("cvss_v3_score", "mean"),
    max_cvss      = ("cvss_v3_score", "max"),
    exploit_count = ("is_kev",        "sum"),
    mean_epss     = ("epss_score",    "mean"),
).reset_index()

# Top 20 CWE one-hot encoding
top_20_cwe  = vulns_raw["cwe_id"].value_counts().head(20).index.tolist()
vulns_cwe   = vulns_raw[vulns_raw["cwe_id"].isin(top_20_cwe)].copy()
cwe_dummies = pd.get_dummies(vulns_cwe[["service_name", "cwe_id"]], columns=["cwe_id"], dtype=int)
cwe_agg     = cwe_dummies.groupby("service_name").sum().reset_index()

# Aggregating Repo Health
repo_agg = repo.groupby("service_name").agg(
    commit_count_90d         = ("commit_count_90d",         "mean"),
    contributor_count_90d    = ("contributor_count_90d",    "mean"),
    open_issues              = ("open_issues",              "mean"),
    security_issues_open     = ("security_issues_open",     "mean"),
    mean_days_close_security = ("mean_days_close_security", "mean"),
    days_since_last_release  = ("days_since_last_release",  "mean"),
    has_security_policy      = ("has_security_policy",      "max"),
).reset_index()

print(f"incidents_agg: {incidents_agg.shape}")
print(f"vuln_agg     : {vuln_agg.shape}")
print(f"cwe_agg      : {cwe_agg.shape}")
print(f"repo_agg     : {repo_agg.shape}")


Unique services: 2733
incidents_agg: (8, 5)
vuln_agg     : (2733, 7)
cwe_agg      : (1783, 21)
repo_agg     : (24, 8)


**2c. Merge, feature engineering & configurable risk label**

In [6]:
# Merging all tables on service_name ─
df = services.copy()
df = df.merge(repo_agg,      on="service_name", how="left")
df = df.merge(incidents_agg, on="service_name", how="left")
df = df.merge(vuln_agg,      on="service_name", how="left")
df = df.merge(cwe_agg,       on="service_name", how="left")

# Safe fillna — numeric columns only
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# Ensuring CWE columns are int
cwe_cols_in_df = [c for c in df.columns if c.startswith("cwe_id_")]
df[cwe_cols_in_df] = df[cwe_cols_in_df].astype(int)

# Derived compound features
df["vuln_density"]  = df["open_vulns"]              / (df["total_vulns"] + 1)
df["risk_velocity"] = df["incident_count"]          * df["recurrence_rate"]
df["patch_debt"]    = df["days_since_last_release"] * df["security_issues_open"]
df["exploit_ratio"] = df["exploit_count"]           / (df["total_vulns"] + 1)
df["epss_risk"]     = df["mean_epss"]               * df["total_vulns"]

# Encoding categorical columns
df["ecosystem"] = df["ecosystem"].fillna("unknown")
df["vendor"]    = df["vendor"].fillna("unknown")
le = LabelEncoder()
df["ecosystem_enc"] = le.fit_transform(df["ecosystem"])
df["vendor_enc"]    = le.fit_transform(df["vendor"])
df["is_oss_enc"]    = pd.to_numeric(df["is_oss"],  errors="coerce").fillna(0).astype(int)
df["is_saas_enc"]   = pd.to_numeric(df["is_saas"], errors="coerce").fillna(0).astype(int)

# Configurable threshold-based risk label
# Thresholds are set in Section 0 — edit there to customize
df["high_risk"] = (
    (df["mean_cvss"]      >= RISK_THRESHOLDS["cvss_min"])    |
    (df["exploit_count"]  >= RISK_THRESHOLDS["kev_min"])     |
    (df["mean_epss"]      >= RISK_THRESHOLDS["epss_min"])    |
    (df["incident_count"] >= RISK_THRESHOLDS["incident_min"])
).astype(int)

print("Feature Engineering complete")
print(f"  DataFrame shape  : {df.shape}")
print(f"  Services (rows)  : {len(df)}")
print(f"  CWE flag cols    : {len(cwe_cols_in_df)}")
print(f"  Null values left : {df[numeric_cols].isna().any().any()}")
print(f"  Label dist       : high_risk=1 → {df['high_risk'].sum()} / {len(df)} ({df['high_risk'].mean():.1%})")
print()
print("  Active thresholds:")
for k, v in RISK_THRESHOLDS.items():
    print(f"    {k:<20} {v}")

df[["service_name", "vendor", "ecosystem", "open_vulns",
    "mean_cvss", "mean_epss", "exploit_count",
    "incident_count", "high_risk"]].sort_values("mean_cvss", ascending=False)


Feature Engineering complete
  DataFrame shape  : (2733, 52)
  Services (rows)  : 2733
  CWE flag cols    : 20
  Null values left : False
  Label dist       : high_risk=1 → 1712 / 2733 (62.6%)

  Active thresholds:
    cvss_min             7.0
    kev_min              1
    epss_min             0.1
    incident_min         3


,service_name,vendor,ecosystem,open_vulns,mean_cvss,mean_epss,exploit_count,incident_count,high_risk
2569,voidzero,voidzero,SaaS,0,10.0,0.0,0,0.0,1
964,ddsn,ddsn,SaaS,0,10.0,0.0,0,0.0,1
2388,scoder,scoder,SaaS,0,10.0,0.0,0,0.0,1
2538,nwclark,nwclark,SaaS,0,10.0,0.0,0,0.0,1
2539,atrodo,atrodo,SaaS,0,10.0,0.0,0,0.0,1
...,...,...,...,...,...,...,...,...,...
2714,marimo,marimo,SaaS,0,0.0,0.0,1,0.0,1
2715,aquasecurity,aquasecurity,SaaS,0,0.0,0.0,1,0.0,1
2730,array networks,array networks,SaaS,0,0.0,0.0,1,0.0,1
2731,openplc,openplc,SaaS,0,0.0,0.0,2,0.0,1


## 3 - Feature Selection & Train/Test Split

In [7]:
CWE_COLS = [c for c in df.columns if c.startswith("cwe_id_")]

# Removed features to prevent leakage
# high_risk label is defined by: mean_cvss, exploit_count, mean_epss, incident_count
# epss_risk = mean_epss * total_vulns  (leaks mean_epss)
# exploit_ratio = exploit_count / total_vulns  (leaks exploit_count)
# risk_velocity = incident_count * recurrence_rate  (leaks incident_count)
LEAKING_FEATURES = {
    "mean_cvss", "max_cvss",        # used directly in high_risk threshold
    "exploit_count",                 # used directly in high_risk threshold
    "mean_epss",                     # used directly in high_risk threshold
    "incident_count",                # used directly in high_risk threshold
    "epss_risk",                     # derived from mean_epss
    "exploit_ratio",                 # derived from exploit_count
    "risk_velocity",                 # derived from incident_count
}

FEATURE_COLS = [
    "total_vulns", "open_vulns", "vuln_density",
    "commit_count_90d", "contributor_count_90d", "open_issues",
    "security_issues_open", "mean_days_close_security",
    "days_since_last_release", "has_security_policy", "patch_debt",
    "avg_severity", "recurrence_rate", "total_downtime",
    "vendor_enc", "ecosystem_enc", "is_oss_enc", "is_saas_enc",
] + CWE_COLS

print(f"⚠ Removed {len(LEAKING_FEATURES)} leaking features: {LEAKING_FEATURES}")

FEATURE_COLS = [f for f in FEATURE_COLS if f in df.columns]

# Mapping each feature to its category weight

# Features in a high-weight category are multiplied by a larger number, the model will pay more attention to them during training.

FEATURE_WEIGHT_MAP = {
    # Vulnerability severity — mean_cvss, max_cvss, mean_epss removed (label leakage)
    # Vulnerability volume
    "open_vulns":                 RISK_WEIGHTS["vulnerability_volume"],
    "total_vulns":                RISK_WEIGHTS["vulnerability_volume"],
    "vuln_density":               RISK_WEIGHTS["vulnerability_volume"],

    # Active exploitation — exploit_count, exploit_ratio, epss_risk removed (label leakage)
    # Incident history — incident_count and risk_velocity removed (label leakage)
    "avg_severity":               RISK_WEIGHTS["incident_history"],
    "recurrence_rate":            RISK_WEIGHTS["incident_history"],
    "total_downtime":             RISK_WEIGHTS["incident_history"],

    # Repository health
    "commit_count_90d":           RISK_WEIGHTS["repository_health"],
    "contributor_count_90d":      RISK_WEIGHTS["repository_health"],
    "open_issues":                RISK_WEIGHTS["repository_health"],
    "security_issues_open":       RISK_WEIGHTS["repository_health"],
    "mean_days_close_security":   RISK_WEIGHTS["repository_health"],
    "days_since_last_release":    RISK_WEIGHTS["repository_health"],
    "has_security_policy":        RISK_WEIGHTS["repository_health"],
    "patch_debt":                 RISK_WEIGHTS["repository_health"],

    # Service metadata — neutral weight (average of all categories)
    "vendor_enc":                 sum(RISK_WEIGHTS.values()) / len(RISK_WEIGHTS),
    "ecosystem_enc":              sum(RISK_WEIGHTS.values()) / len(RISK_WEIGHTS),
    "is_oss_enc":                 sum(RISK_WEIGHTS.values()) / len(RISK_WEIGHTS),
    "is_saas_enc":                sum(RISK_WEIGHTS.values()) / len(RISK_WEIGHTS),
}

# Build weight vector — CWE columns use cwe_attack_types weight
feature_weights = np.array([
    FEATURE_WEIGHT_MAP.get(f, RISK_WEIGHTS["cwe_attack_types"])
    for f in FEATURE_COLS
])

# Applying weights to feature matrix
X_raw      = df[FEATURE_COLS]
X_weighted = X_raw * feature_weights          # each column scaled by its weight
y          = df["high_risk"]

print("Feature weighting applied")
print(f"  Total features     : {len(FEATURE_COLS)}  ({len(CWE_COLS)} CWE columns)")
print()
print("  FEATURE WEIGHT VECTOR (top 15 by weight)")
print("  " + "─" * 55)
weight_series = pd.Series(feature_weights, index=FEATURE_COLS).sort_values(ascending=False)
for feat, w in weight_series.head(15).items():
    bar = "█" * int(w * 200)
    print(f"  {feat:<35} {bar:<10} {w:.3f}")

# Train/test split on WEIGHTED features
if len(df) >= 20:
    X_train, X_test, y_train, y_test = train_test_split(
        X_weighted, y, test_size=0.3, random_state=42, stratify=y
    )
else:
    X_train, X_test, y_train, y_test = X_weighted, X_weighted, y, y
    print("\n⚠ Small dataset — using full data for training; LOO CV is primary metric.")

print(f"\n  Training rows : {len(X_train)}")
print(f"  Test rows     : {len(X_test)}")
print(f"  Class balance : {y_train.mean():.1%} high risk in training set")
print()
print("  Active category weights:")
for cat, w in RISK_WEIGHTS.items():
    print(f"    {cat:<25} {w:.0%}")


⚠ Removed 8 leaking features: {'mean_epss', 'risk_velocity', 'max_cvss', 'mean_cvss', 'exploit_ratio', 'incident_count', 'epss_risk', 'exploit_count'}
Feature weighting applied
  Total features     : 38  (20 CWE columns)

  FEATURE WEIGHT VECTOR (top 15 by weight)
  ───────────────────────────────────────────────────────
  total_vulns                         ██████████████████████████████████████████████████ 0.250
  open_vulns                          ██████████████████████████████████████████████████ 0.250
  vuln_density                        ██████████████████████████████████████████████████ 0.250
  ecosystem_enc                       █████████████████████████████████ 0.167
  is_oss_enc                          █████████████████████████████████ 0.167
  is_saas_enc                         █████████████████████████████████ 0.167
  vendor_enc                          █████████████████████████████████ 0.167
  recurrence_rate                     ████████████████████ 0.100
  avg_severity 

## 4 · Model Definitions

Stronger regularisation applied across all models (v2 anti-overfitting fix).

In [8]:
# Model 1: Logistic Regression
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000, class_weight="balanced", C=0.01, random_state=42
    ))
])

# Model 2: Random Forest
rf = RandomForestClassifier(
    n_estimators=100, max_depth=3, max_features="sqrt",
    min_samples_leaf=3, class_weight="balanced", random_state=42, n_jobs=-1
)

# Model 3: XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=2, learning_rate=0.05,
    subsample=0.6, colsample_bytree=0.5,
    reg_alpha=1.0, reg_lambda=2.0, min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum() if y_train.sum() > 0 else 1,
    eval_metric="logloss", use_label_encoder=False, random_state=42, verbosity=0,
)

# Model 4: Neural Network
nn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(32, 16), activation="relu", solver="adam",
        alpha=0.1, max_iter=500, early_stopping=True,
        validation_fraction=0.2, random_state=42
    ))
])

# Model 5: Isolation Forest
contamination = min(max(float(y.mean()), 0.05), 0.5)
iso_forest = IsolationForest(
    n_estimators=100, contamination=contamination,
    max_samples="auto", random_state=42
)

models = {
    "Logistic Regression": lr,
    "Random Forest":       rf,
    "XGBoost":             xgb_model,
    "Neural Network":      nn,
}

print("All 5 models defined")
print(f"  Isolation Forest contamination: {contamination:.2f}")


All 5 models defined
  Isolation Forest contamination: 0.50


## 5 - Training & Evaluation

In [9]:
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Training with 5-Fold Stratified cross-validation.
# Models are trained on WEIGHTED features.

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc  = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else float("nan")
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    try:
        # 5-fold CV on weighted features
        cv_scores = cross_val_score(model, X_weighted, y, cv=cv, scoring="roc_auc", n_jobs=-1)
        loo_auc   = cv_scores.mean()
        loo_std   = cv_scores.std()
    except Exception:
        loo_auc, loo_std = float("nan"), float("nan")

    fpr, tpr, _ = roc_curve(y_test, y_proba) if len(np.unique(y_test)) > 1 else ([0,1],[0,1],[])

    results[name] = dict(
        model=model, y_pred=y_pred, y_proba=y_proba,
        auc=auc, loo_auc=loo_auc, loo_std=loo_std,
        precision=prec, recall=rec, f1=f1, fpr=fpr, tpr=tpr
    )

    print(f"{'─'*55}")
    print(f" {name}")
    print(f"  AUC-ROC (test)   : {auc:.4f}")
    print(f"  AUC-ROC (5-Fold CV) : {loo_auc:.4f} ± {loo_std:.4f}  ← primary metric")
    print(f"  Precision: {prec:.4f}   Recall: {rec:.4f}   F1: {f1:.4f}")
    if len(np.unique(y_test)) > 1:
        print(classification_report(y_test, y_pred, target_names=["Low risk", "High risk"]))


───────────────────────────────────────────────────────
 Logistic Regression
  AUC-ROC (test)   : 0.5447
  AUC-ROC (5-Fold CV) : 0.6021 ± 0.0448  ← primary metric
  Precision: 0.6327   Recall: 0.4591   F1: 0.5321
              precision    recall  f1-score   support

    Low risk       0.38      0.55      0.45       306
   High risk       0.63      0.46      0.53       514

    accuracy                           0.49       820
   macro avg       0.51      0.51      0.49       820
weighted avg       0.54      0.49      0.50       820

───────────────────────────────────────────────────────
 Random Forest
  AUC-ROC (test)   : 0.7219
  AUC-ROC (5-Fold CV) : 0.7131 ± 0.0249  ← primary metric
  Precision: 0.7099   Recall: 0.8949   F1: 0.7917
              precision    recall  f1-score   support

    Low risk       0.69      0.39      0.49       306
   High risk       0.71      0.89      0.79       514

    accuracy                           0.70       820
   macro avg       0.70      0.64  

In [10]:
# Isolation Forest — also trained on weighted features
iso_forest.fit(X_train)
iso_raw   = iso_forest.decision_function(X_test)
iso_score = -iso_raw
iso_score = (iso_score - iso_score.min()) / (iso_score.max() - iso_score.min() + 1e-9)
iso_pred  = (iso_score >= 0.5).astype(int)

iso_auc  = roc_auc_score(y_test, iso_score) if len(np.unique(y_test)) > 1 else float("nan")
iso_prec = precision_score(y_test, iso_pred, zero_division=0)
iso_rec  = recall_score(y_test, iso_pred, zero_division=0)
iso_f1   = f1_score(y_test, iso_pred, zero_division=0)
iso_fpr, iso_tpr, _ = roc_curve(y_test, iso_score) if len(np.unique(y_test)) > 1 else ([0,1],[0,1],[])

results["Isolation Forest"] = dict(
    model=iso_forest, y_pred=iso_pred, y_proba=iso_score,
    auc=iso_auc, loo_auc=None, loo_std=None,
    precision=iso_prec, recall=iso_rec, f1=iso_f1,
    fpr=iso_fpr, tpr=iso_tpr
)

print(f"Isolation Forest — AUC: {iso_auc:.4f}  Precision: {iso_prec:.4f}  Recall: {iso_rec:.4f}  F1: {iso_f1:.4f}")


Isolation Forest — AUC: 0.5135  Precision: 0.6957  Recall: 0.0311  F1: 0.0596


## 6 - Feature Importance from XGBoost

In [11]:
CWE_LABEL_MAP = {
    "CWE-20":  "Improper Input Validation",
    "CWE-22":  "Path Traversal",
    "CWE-74":  "Injection",
    "CWE-78":  "OS Command Injection",
    "CWE-79":  "Cross-Site Scripting (XSS)",
    "CWE-89":  "SQL Injection",
    "CWE-90":  "LDAP Injection",
    "CWE-94":  "Code Injection",
    "CWE-119": "Buffer Overflow",
    "CWE-125": "Out-of-Bounds Read",
    "CWE-190": "Integer Overflow",
    "CWE-200": "Information Exposure",
    "CWE-269": "Improper Privilege Management",
    "CWE-276": "Incorrect Default Permissions",
    "CWE-287": "Authentication Bypass",
    "CWE-295": "Improper Certificate Validation",
    "CWE-306": "Missing Authentication",
    "CWE-310": "Cryptographic Issues",
    "CWE-319": "Cleartext Transmission",
    "CWE-326": "Inadequate Encryption Strength",
    "CWE-327": "Use of Broken Algorithm",
    "CWE-330": "Insufficient Random Values",
    "CWE-338": "Weak PRNG",
    "CWE-352": "Cross-Site Request Forgery (CSRF)",
    "CWE-362": "Race Condition",
    "CWE-369": "Divide by Zero",
    "CWE-400": "Uncontrolled Resource Consumption",
    "CWE-416": "Use After Free",
    "CWE-434": "Unrestricted File Upload",
    "CWE-476": "Null Pointer Dereference",
    "CWE-502": "Deserialization of Untrusted Data",
    "CWE-601": "Open Redirect",
    "CWE-611": "XML External Entity (XXE)",
    "CWE-639": "Insecure Direct Object Reference",
    "CWE-787": "Out-of-Bounds Write",
    "CWE-798": "Hard-coded Credentials",
    "CWE-862": "Missing Authorization",
    "CWE-863": "Incorrect Authorization",
    "CWE-918": "Server-Side Request Forgery (SSRF)",
    "CWE-943": "NoSQL Injection",
}

FEATURE_LABEL_MAP = {
    "total_vulns":               "Total Vulnerabilities",
    "open_vulns":                "Open High/Critical Vulns",
    "vuln_density":              "Vulnerability Density",
    "commit_count_90d":          "Commits (90d)",
    "contributor_count_90d":     "Contributors (90d)",
    "open_issues":               "Open Issues",
    "security_issues_open":      "Open Security Issues",
    "mean_days_close_security":  "Avg Days to Close Security Issue",
    "days_since_last_release":   "Days Since Last Release",
    "has_security_policy":       "Has Security Policy",
    "patch_debt":                "Patch Debt Score",
    "avg_severity":              "Avg Incident Severity",
    "recurrence_rate":           "Unresolved Incident Rate",
    "total_downtime":            "Total Downtime (min)",
    "vendor_enc":                "Vendor",
    "ecosystem_enc":             "Ecosystem",
    "is_oss_enc":                "Is OSS",
    "is_saas_enc":               "Is SaaS",
}

def readable_feature_name(col):
    if col.startswith("cwe_id_"):
        raw = col.replace("cwe_id_", "")
        parts = raw.split(",")
        labels = [CWE_LABEL_MAP.get(p.strip(), p.strip()) for p in parts]
        return " + ".join(labels)
    return FEATURE_LABEL_MAP.get(col, col)

importances = pd.Series(
    xgb_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False).head(15)

importances_named = importances.copy()
importances_named.index = [readable_feature_name(f) for f in importances.index]

print("Top 15 Feature Importances (XGBoost gain)")
print("Note: importances reflect WEIGHTED features — high-weight categories")
print("will tend to rank higher because their features were amplified during training.\n")
print(importances_named.to_string())

Top 15 Feature Importances (XGBoost gain)
Note: importances reflect WEIGHTED features — high-weight categories
will tend to rank higher because their features were amplified during training.

Cross-Site Scripting (XSS)            0.210051
OS Command Injection                  0.091054
Total Vulnerabilities                 0.067151
Code Injection                        0.059198
SQL Injection                         0.058307
Cross-Site Request Forgery (CSRF)     0.054060
Missing Authentication                0.050115
Out-of-Bounds Read                    0.047012
Deserialization of Untrusted Data     0.043919
Missing Authorization                 0.043081
Incorrect Authorization               0.041120
Information Exposure                  0.039568
Insecure Direct Object Reference      0.038306
Out-of-Bounds Write                   0.036486
Server-Side Request Forgery (SSRF)    0.035578


## 7 · Service-Level Risk Scores

In [12]:
# ── Score all services using weighted features ───────────────────────────────
df["xgb_risk_score"] = xgb_model.predict_proba(X_weighted)[:, 1]
df["lr_risk_score"]  = lr.predict_proba(X_weighted)[:, 1]

iso_full_raw   = iso_forest.decision_function(X_weighted)
iso_full_score = -iso_full_raw
iso_full_score = (iso_full_score - iso_full_score.min()) / (iso_full_score.max() - iso_full_score.min() + 1e-9)
df["anomaly_score"] = iso_full_score
df["anomaly_flag"]  = (iso_full_score >= 0.5).astype(int)

# ── Configurable risk bands ───────────────────────────────────────────────────
df["risk_level"] = pd.cut(
    df["lr_risk_score"],
    bins=[0, RISK_BANDS["medium_threshold"], RISK_BANDS["high_threshold"], 1.0],
    labels=["Low", "Medium", "High"]
)

top_risk = (
    df[["service_name", "vendor", "ecosystem", "open_vulns",
        "mean_cvss", "mean_epss", "exploit_count", "incident_count",
        "lr_risk_score", "anomaly_score", "anomaly_flag", "risk_level"]]
    .sort_values("lr_risk_score", ascending=False)
    .reset_index(drop=True)
)

print(f"Scores generated using Option A weighted features ✓")
print(f"  Risk bands:  High ≥ {RISK_BANDS['high_threshold']}  |  Medium ≥ {RISK_BANDS['medium_threshold']}")
print(f"  High risk services  : {(df['risk_level'] == 'High').sum()}")
print(f"  Medium risk services: {(df['risk_level'] == 'Medium').sum()}")
print(f"  Low risk services   : {(df['risk_level'] == 'Low').sum()}")
print()
top_risk


Scores generated using Option A weighted features ✓
  Risk bands:  High ≥ 0.7  |  Medium ≥ 0.4
  High risk services  : 19
  Medium risk services: 2711
  Low risk services   : 3



,service_name,vendor,ecosystem,open_vulns,mean_cvss,mean_epss,exploit_count,incident_count,lr_risk_score,anomaly_score,anomaly_flag,risk_level
0,amazon,amazon,SaaS,0,7.113725,0.00000,0,0.0,1.000000,0.745855,1,High
1,snowflake,snowflake,SaaS,0,7.233333,0.00015,0,22.0,1.000000,0.456844,0,High
2,google,google,SaaS,0,7.198424,0.00000,12,0.0,0.999858,0.969441,1,High
3,unknown,unknown,SaaS,0,6.637114,0.00000,4,0.0,0.999811,1.000000,1,High
4,elastic,elastic,SaaS,0,6.404167,0.00000,0,37.0,0.998141,0.682818,1,High
...,...,...,...,...,...,...,...,...,...,...,...,...
2728,openwebui,openwebui,SaaS,0,6.636232,0.00000,0,0.0,0.422147,0.647646,1,Medium
2729,hcltech,hcltech,SaaS,0,6.507317,0.00000,0,0.0,0.421772,0.549373,1,Medium
2730,mattermost,mattermost,SaaS,0,5.324000,0.00000,0,0.0,0.351380,0.675822,1,Low
2731,openclaw,openclaw,SaaS,0,7.075238,0.00000,0,0.0,0.304960,0.824362,1,Low


In [13]:
model_names = list(results.keys())
sup_names   = [n for n in model_names if n != "Isolation Forest"]

## 8 - Final Model Selection

In [14]:
rf_auc = results["Random Forest"]["loo_auc"]
xgb_auc = results["XGBoost"]["loo_auc"]

# If the difference in CV AUC is less than 1%, default to XGBoost
if abs(rf_auc - xgb_auc) < 0.01:
    best_sup = "XGBoost"
else:
    best_sup = max(sup_names, key=lambda n: results[n]["loo_auc"] if results[n]["loo_auc"] and not np.isnan(results[n]["loo_auc"]) else 0)

print("_" * 60)
print("FINAL MODEL SELECTION)")
print("_" * 60)
print(f"  Best model (CV AUC): {best_sup}")
print(f"  CV AUC : {results[best_sup]['loo_auc']:.4f} \u00b1 {results[best_sup]['loo_std']:.4f}")
print(f"  F1      : {results[best_sup]['f1']:.4f}")
print()
print("  Score Generation:")
print("  Models trained on features weighted by your configuration.")
print("  High-weight categories had amplified features; therefore the model")
print("  learned to treat them as more important signal.")
print()
print("  Active Weight Configuration:")
for cat, w in RISK_WEIGHTS.items():
    bar = "\u2588" * int(w * 40)
    print(f"    {cat:<25} {bar:<40} {w:.0%}")
print()
print("  Active Thresholds (used for high_risk label):")
for k, v in RISK_THRESHOLDS.items():
    print(f"    {k:<20} {v}")
print()

summary = pd.DataFrame({
    "Model":    model_names,
    "CV AUC":  [f"{results[n]['loo_auc']:.4f}" if results[n]["loo_auc"] and not np.isnan(results[n]["loo_auc"]) else "N/A" for n in model_names],
    "Test AUC": [f"{results[n]['auc']:.4f}"     if not np.isnan(results[n]["auc"]) else "N/A" for n in model_names],
    "F1":       [f"{results[n]['f1']:.4f}"       for n in model_names],
})
print(summary.to_string(index=False))

____________________________________________________________
FINAL MODEL SELECTION)
____________________________________________________________
  Best model (CV AUC): XGBoost
  CV AUC : 0.7134 ± 0.0264
  F1      : 0.7867

  Score Generation:
  Models trained on features weighted by your configuration.
  High-weight categories had amplified features; therefore the model
  learned to treat them as more important signal.

  Active Weight Configuration:
    vulnerability_severity    ████████████████                         40%
    vulnerability_volume      ██████████                               25%
    active_exploitation       ██████                                   15%
    incident_history          ████                                     10%
    repository_health         ██                                       5%
    cwe_attack_types          ██                                       5%

  Active Thresholds (used for high_risk label):
    cvss_min             7.0
    kev_min        

## 9 - Service Risk Drill-Down

In [15]:
SERVICE_FILTER = input("Enter service name to analyse (or press Enter for all): ").strip()

if not SERVICE_FILTER:
    SERVICE_FILTER = None
    print("No filter applied.")
else:
    matches = df[df["service_name"].str.lower().str.contains(SERVICE_FILTER.lower(), na=False)]["service_name"].unique()
    if len(matches) == 0:
        print(f"\nNo services found matching '{SERVICE_FILTER}'.")
        print("Available services:")
        for name in sorted(df["service_name"].dropna().unique()):
            print(f"  - {name}")
        SERVICE_FILTER = None
    elif len(matches) == 1:
        print(f"Found: {matches[0]}")
    else:
        print(f"Found {len(matches)} matching services:")
        for name in matches:
            print(f"  - {name}")


Enter service name to analyse (or press Enter for all): salesforce
Found: salesforce


In [16]:
if SERVICE_FILTER:
    mask = df["service_name"].str.lower().str.contains(SERVICE_FILTER.lower(), na=False)
    svc  = df[mask].copy()

    if svc.empty:
        print(f"No service found matching '{SERVICE_FILTER}'")
    else:
        CWE_LABELS = {
    "CWE-20":  "Improper Input Validation",
    "CWE-22":  "Path Traversal",
    "CWE-74":  "Injection",
    "CWE-78":  "OS Command Injection",
    "CWE-79":  "Cross-Site Scripting (XSS)",
    "CWE-89":  "SQL Injection",
    "CWE-90":  "LDAP Injection",
    "CWE-94":  "Code Injection",
    "CWE-119": "Buffer Overflow",
    "CWE-125": "Out-of-Bounds Read",
    "CWE-190": "Integer Overflow",
    "CWE-200": "Information Exposure",
    "CWE-269": "Improper Privilege Management",
    "CWE-276": "Incorrect Default Permissions",
    "CWE-287": "Authentication Bypass",
    "CWE-295": "Improper Certificate Validation",
    "CWE-306": "Missing Authentication",
    "CWE-310": "Cryptographic Issues",
    "CWE-319": "Cleartext Transmission",
    "CWE-326": "Inadequate Encryption Strength",
    "CWE-327": "Use of Broken Algorithm",
    "CWE-330": "Insufficient Random Values",
    "CWE-338": "Weak PRNG",
    "CWE-352": "Cross-Site Request Forgery (CSRF)",
    "CWE-362": "Race Condition",
    "CWE-369": "Divide by Zero",
    "CWE-400": "Uncontrolled Resource Consumption",
    "CWE-416": "Use After Free",
    "CWE-434": "Unrestricted File Upload",
    "CWE-476": "Null Pointer Dereference",
    "CWE-502": "Deserialization of Untrusted Data",
    "CWE-601": "Open Redirect",
    "CWE-611": "XML External Entity (XXE)",
    "CWE-639": "Insecure Direct Object Reference",
    "CWE-787": "Out-of-Bounds Write",
    "CWE-798": "Hard-coded Credentials",
    "CWE-862": "Missing Authorization",
    "CWE-863": "Incorrect Authorization",
    "CWE-918": "Server-Side Request Forgery (SSRF)",
    "CWE-943": "NoSQL Injection",
        }
        FEATURE_LABELS = {
     "mean_cvss":                 "Average CVSS Score",
     "max_cvss":                  "Highest CVSS Score",
      "mean_epss":                 "Avg Exploitation Probability (EPSS)",
      "open_vulns":                "Open High/Critical Vulnerabilities",
      "total_vulns":               "Total Vulnerabilities",
      "vuln_density":              "Vulnerability Density",
      "exploit_count":             "Known Exploited CVEs (KEV)",
      "exploit_ratio":             "Exploit Ratio",
      "epss_risk":                 "Overall Exploitation Exposure",
      "incident_count":            "Total Incidents",
      "avg_severity":              "Average Incident Severity",
      "recurrence_rate":           "Unresolved Incident Rate",
      "total_downtime":            "Total Downtime (minutes)",
      "risk_velocity":             "Incident Risk Velocity",
      "security_issues_open":      "Open Security Issues",
      "days_since_last_release":   "Days Since Last Release",
      "mean_days_close_security":  "Avg Days to Close Security Issue",
      "patch_debt":                "Patch Debt Score",
        }

        # Scoring using WEIGHTED features (same as training)
        svc_weighted = svc[FEATURE_COLS] * feature_weights
        svc["lr_risk_score"] = lr.predict_proba(svc_weighted)[:, 1]
        iso_raw  = iso_forest.decision_function(svc_weighted)
        iso_s    = -iso_raw
        iso_s    = (iso_s - iso_s.min()) / (iso_s.max() - iso_s.min() + 1e-9)
        svc["anomaly_score"] = iso_s
        svc["anomaly_flag"]  = (iso_s >= 0.5).astype(int)
        svc["risk_level"]    = pd.cut(
            svc["lr_risk_score"],
            bins=[0, RISK_BANDS["medium_threshold"], RISK_BANDS["high_threshold"], 1.0],
            labels=["Low", "Medium", "High"]
        )

        print("=" * 70)
        print(f"  RISK PROFILE: {SERVICE_FILTER.upper()}")
        print("=" * 70)

        for _, row in svc.iterrows():
            print(f"\n  Service       : {row['service_name']}")
            print(f"  Vendor        : {row['vendor']}")
            print(f"  Ecosystem     : {row['ecosystem']}")
            print(f"  Risk Score    : {row['lr_risk_score']:.4f}  ({row['risk_level']})")
            print(f"  Anomaly Score : {row['anomaly_score']:.4f}  ({'⚠ FLAGGED' if row['anomaly_flag'] == 1 else 'Normal'})")

        # Weighted risk areas
        RISK_AREAS = {
            "Vulnerability Severity": {
                "weight": RISK_WEIGHTS["vulnerability_severity"],
                "features": [
                    ("mean_cvss",  svc["mean_cvss"].values[0]  / 10),
                    ("max_cvss",   svc["max_cvss"].values[0]   / 10),
                    ("mean_epss",  svc["mean_epss"].values[0]),
                ]
            },
            "Vulnerability Volume": {
                "weight": RISK_WEIGHTS["vulnerability_volume"],
                "features": [
                    ("open_vulns",   svc["open_vulns"].values[0]   / (df["open_vulns"].max() + 1)),
                    ("total_vulns",  svc["total_vulns"].values[0]  / (df["total_vulns"].max() + 1)),
                    ("vuln_density", svc["vuln_density"].values[0]),
                ]
            },
            "Active Exploitation": {
                "weight": RISK_WEIGHTS["active_exploitation"],
                "features": [
                    ("exploit_count", svc["exploit_count"].values[0] / (df["exploit_count"].max() + 1)),
                    ("exploit_ratio", svc["exploit_ratio"].values[0]),
                    ("epss_risk",     svc["epss_risk"].values[0]     / (df["epss_risk"].max() + 1)),
                ]
            },
            "Incident History": {
                "weight": RISK_WEIGHTS["incident_history"],
                "features": [
                    ("incident_count",  svc["incident_count"].values[0]  / (df["incident_count"].max() + 1)),
                    ("avg_severity",    svc["avg_severity"].values[0]    / 4),
                    ("recurrence_rate", svc["recurrence_rate"].values[0]),
                    ("total_downtime",  svc["total_downtime"].values[0]  / (df["total_downtime"].max() + 1)),
                ]
            },
            "Repository Health": {
                "weight": RISK_WEIGHTS["repository_health"],
                "features": [
                    ("security_issues_open",     svc["security_issues_open"].values[0]     / (df["security_issues_open"].max() + 1)),
                    ("days_since_last_release",  svc["days_since_last_release"].values[0]  / (df["days_since_last_release"].max() + 1)),
                    ("mean_days_close_security", svc["mean_days_close_security"].values[0] / (df["mean_days_close_security"].max() + 1)),
                    ("patch_debt",               svc["patch_debt"].values[0]               / (df["patch_debt"].max() + 1)),
                ]
            },
            "CWE Attack Types": {
                "weight": RISK_WEIGHTS["cwe_attack_types"],
                "features": [
                    (CWE_LABELS.get(col.replace("cwe_id_", ""), col.replace("cwe_id_", "")), float(svc[col].values[0]))
                    for col in FEATURE_COLS if col.startswith("cwe_id_") and svc[col].values[0] > 0
                ]
            },
        }

        # Computing weighted category scores
        category_scores = {}
        weighted_total  = 0.0
        for category, data in RISK_AREAS.items():
            features = data["features"]
            weight   = data["weight"]
            raw = sum(v for _, v in features) / len(features) if features else 0.0
            category_scores[category] = {
                "raw":      min(raw, 1.0),
                "weighted": min(raw, 1.0) * weight,
                "weight":   weight,
            }
            weighted_total += category_scores[category]["weighted"]

        # Print breakdown
        print(f"\n{'─'*70}")
        print("  RISK AREA BREAKDOWN  (weighted by your Section 0b configuration)")
        print(f"{'─'*70}")
        print(f"  {'Category':<25} {'Weight':>6}  {'Score':>5}  {'Wtd':>5}  Bar")
        print(f"  {'─'*25} {'─'*6}  {'─'*5}  {'─'*5}  {'─'*25}")

        sorted_areas = sorted(category_scores.items(), key=lambda x: x[1]["weighted"], reverse=True)
        BAR_WIDTH = 25

        for area, scores in sorted_areas:
            raw      = scores["raw"]
            weighted = scores["weighted"]
            weight   = scores["weight"]
            filled   = int(raw * BAR_WIDTH)
            bar      = "█" * filled + "░" * (BAR_WIDTH - filled)
            level    = "HIGH  ⚠" if raw >= RISK_BANDS["high_threshold"] else \
                       "MEDIUM " if raw >= RISK_BANDS["medium_threshold"] else "LOW   ✓"
            print(f"  {area:<25} {weight:>5.0%}  {raw:>5.2f}  {weighted:>5.2f}  [{bar}]  {level}")

        print(f"  {'─'*25} {'─'*6}  {'─'*5}  {'─'*5}")
        print(f"  {'WEIGHTED TOTAL':<25} {'100%':>6}         {weighted_total:>5.2f}")

        # Top risk drivers
        print(f"\n{'─'*70}")
        print("  TOP RISK DRIVERS  (sorted by weighted contribution)")
        print(f"{'─'*70}")

        all_features = []
        for category, data in RISK_AREAS.items():
            w = data["weight"]
            for feat_name, feat_score in data["features"]:
                all_features.append((category, feat_name, feat_score, feat_score * w))

        top_features = sorted(all_features, key=lambda x: x[3], reverse=True)[:10]

        for i, (cat, feat, score, wtd_score) in enumerate(top_features, 1):
            level        = "⚠ HIGH  " if score >= RISK_BANDS["high_threshold"] else \
                           "~ MEDIUM" if score >= RISK_BANDS["medium_threshold"] else "✓ LOW   "
            display_feat = FEATURE_LABELS.get(feat, feat)
            print(f"  {i:>2}. {level}  {display_feat:<40} {score:.3f}  (wtd: {wtd_score:.3f})  [{cat}]")

        # CWE present
        cwe_present = [col.replace("cwe_id_", "") for col in FEATURE_COLS
                       if col.startswith("cwe_id_") and svc[col].values[0] > 0]

        if cwe_present:
            print(f"\n{'─'*70}")
            print("  CWE ATTACK TYPES PRESENT")
            print(f"{'─'*70}")
            for cwe in cwe_present:
                print(f"  {cwe:<12}  {CWE_LABELS.get(cwe, 'Other')}")

        print(f"\n{'='*70}")

else:
    print("No service selected — skipping drill-down.")
    print("Re-run the cell above and enter a service name.")


  RISK PROFILE: SALESFORCE

  Service       : salesforce
  Vendor        : salesforce
  Ecosystem     : SaaS
  Risk Score    : 0.6796  (Medium)
  Anomaly Score : 0.0000  (Normal)

──────────────────────────────────────────────────────────────────────
  RISK AREA BREAKDOWN  (weighted by your Section 0b configuration)
──────────────────────────────────────────────────────────────────────
  Category                  Weight  Score    Wtd  Bar
  ───────────────────────── ──────  ─────  ─────  ─────────────────────────
  Vulnerability Severity      40%   0.61   0.24  [███████████████░░░░░░░░░░]  MEDIUM 
  CWE Attack Types             5%   1.00   0.05  [█████████████████████████]  HIGH  ⚠
  Vulnerability Volume        25%   0.00   0.00  [░░░░░░░░░░░░░░░░░░░░░░░░░]  LOW   ✓
  Repository Health            5%   0.00   0.00  [░░░░░░░░░░░░░░░░░░░░░░░░░]  LOW   ✓
  Active Exploitation         15%   0.00   0.00  [░░░░░░░░░░░░░░░░░░░░░░░░░]  LOW   ✓
  Incident History            10%   0.00   0.00  [░